# Check if all offsets are present for each variant

For each variant (seq ID) in the hashfrag-filtered FASTA, verify that all 11 offsets × 2 alleles (ref/alt) = 22 entries are present.

In [ ]:
from collections import defaultdict

FASTA = "/scratch/st-cdeboer-1/sambina/position_mpra/outputs/2-opentargets_model_variant_effect/variants_200bp_offsets_hashfrag_filtered.fa"

offsets_per_variant = defaultdict(set)

with open(FASTA) as f:
    for line in f:
        if line.startswith('>'):
            header = line.strip()[1:]
            variant_id, offset_allele = header.split('_offset')
            offset, allele = offset_allele.rsplit('_', 1)
            offsets_per_variant[variant_id].add((offset, allele))

print(f"Total variants: {len(offsets_per_variant)}")

In [ ]:
expected_offsets = {'-90', '-80', '-60', '-40', '-20', '0', '20', '40', '60', '80', '90'}
expected_alleles = {'ref', 'alt'}
expected = {(o, a) for o in expected_offsets for a in expected_alleles}

missing = {}
extra = {}
for v, entries in offsets_per_variant.items():
    m = expected - entries
    e = entries - expected
    if m:
        missing[v] = sorted(m)
    if e:
        extra[v] = sorted(e)

print(f"Expected entries per variant: {len(expected)} ({len(expected_offsets)} offsets x 2 alleles)")
print(f"Variants missing some offsets: {len(missing)}")
print(f"Variants with unexpected extra offsets: {len(extra)}")

if missing:
    print("\nVariants with missing offsets:")
    for v, m in missing.items():
        print(f"  {v}: {m}")
else:
    print("\nAll variants have all expected offsets.")

In [ ]:
# Summary table: count of entries per variant
counts = {v: len(entries) for v, entries in offsets_per_variant.items()}
from collections import Counter
count_dist = Counter(counts.values())
print("Distribution of entry counts per variant:")
for n, freq in sorted(count_dist.items()):
    print(f"  {n} entries: {freq} variants")